# Spot-Check Modal Evaluation Runs

**Environment:** Run this notebook in the `cs224r-hw2` conda environment.

This notebook downloads output files from a Modal Volume and lets you inspect questions, responses, and scores for any evaluation run (AIME, MATH, or GSM8K).

## How to use
1. Set `DATASET`, `MODEL`, and `OUTPUT_TAG` in **Cell 2** to match your run.
2. Run all cells in order (`Run All`).
3. Scroll to the bottom to inspect individual problems.

In [4]:
import json
import os
import subprocess
from pathlib import Path

import pandas as pd
import numpy as np

# Verify Modal CLI is available
result = subprocess.run(["which", "modal"], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("Modal CLI not found. Install with: pip install modal")

print("Modal CLI found:", result.stdout.strip())

Modal CLI found: /home/chung/miniconda3/envs/cs224r-hw2/bin/modal


## 1. Configuration — Edit these variables to match your run

In [5]:
# Which dataset / run do you want to inspect?
DATASET = "gsm8k"          # 'aime' | 'math' | 'gsm8k'
MODEL = "qwen"            # 'qwen' | 'e3'
OUTPUT_TAG = "n4_l32768"  # e.g. 'n2_l32768', 'smoke', 'n16_l32768', etc.

# Volume settings
VOLUME_NAME = "e3-generation-vol"
VOLUME_BASE_DIR = "aime_eval"  # both modal_eval_aime.py and modal_eval_general.py write here

# Where to download files locally
LOCAL_DOWNLOAD_DIR = Path("./modal_spotcheck_downloads")

# For AIME runs from modal_eval_aime.py, the parquet prefix is 'aime2025' instead of 'aime'.
# Set to True if your run used modal_eval_aime.py (not modal_eval_general.py).
IS_LEGACY_AIME = (DATASET == "aime" and not OUTPUT_TAG.startswith("n"))
# ^ heuristic: legacy runs often use custom tags like 'smoke'; general.py uses 'n{k}_l{len}' by default.
# If unsure, you can override the parquet prefix manually below.

PARQUET_PREFIX = DATASET  # general.py uses dataset name as prefix
if DATASET == "aime" and IS_LEGACY_AIME:
    PARQUET_PREFIX = "aime2025"

print(f"Dataset: {DATASET}")
print(f"Model: {MODEL}")
print(f"Tag: {OUTPUT_TAG}")
print(f"Parquet prefix: {PARQUET_PREFIX}")

Dataset: gsm8k
Model: qwen
Tag: n4_l32768
Parquet prefix: gsm8k


## 2. Download files from Modal Volume

In [9]:
LOCAL_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Build remote file names
# Note: modal_eval_general.py includes dataset_key in metrics/per_problem filenames;
#       modal_eval_aime.py does NOT include dataset_key.
if IS_LEGACY_AIME:
    metrics_name = f"metrics_{MODEL}_{OUTPUT_TAG}.json"
    per_problem_name = f"per_problem_{MODEL}_{OUTPUT_TAG}.csv"
else:
    metrics_name = f"metrics_{DATASET}_{MODEL}_{OUTPUT_TAG}.json"
    per_problem_name = f"per_problem_{DATASET}_{MODEL}_{OUTPUT_TAG}.csv"

parquet_name = f"{PARQUET_PREFIX}_{MODEL}_{OUTPUT_TAG}_outputs.parquet"

remote_files = [
    f"{VOLUME_BASE_DIR}/{metrics_name}",
    f"{VOLUME_BASE_DIR}/{per_problem_name}",
    f"{VOLUME_BASE_DIR}/{parquet_name}",
]

local_paths = {}
for remote_path in remote_files:
    local_path = LOCAL_DOWNLOAD_DIR / os.path.basename(remote_path)
    cmd = ["modal", "volume", "get", "--force", VOLUME_NAME, remote_path, str(local_path)]
    print(f"Downloading {os.path.basename(remote_path)} ...")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  ERROR: {result.stderr}")
    else:
        print(f"  -> {local_path}")
        local_paths[os.path.basename(remote_path)] = local_path

if not local_paths:
    raise RuntimeError("No files were downloaded. Check your VOLUME_NAME, DATASET, MODEL, and OUTPUT_TAG.")

  -> modal_spotcheck_downloads/metrics_gsm8k_qwen_n4_l32768.json
  -> modal_spotcheck_downloads/per_problem_gsm8k_qwen_n4_l32768.csv
  -> modal_spotcheck_downloads/gsm8k_qwen_n4_l32768_outputs.parquet


## 3. Load & Summarize Metrics

In [10]:
# Load metrics JSON
metrics_path = local_paths.get(metrics_name)
if metrics_path and metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)
    print("=== Eval Summary ===")
    for k, v in metrics.items():
        print(f"  {k}: {v}")
else:
    print(f"Metrics file not found: {metrics_name}")

=== Eval Summary ===
  dataset: gsm8k
  model: qwen
  tag: n4_l32768
  scorer: gsm8k_flexible
  num_problems: 100
  n_samples: 4
  extract_failures: 0
  pass@1_mean: 0.8975
  pass@1: 0.8975
  pass@4: 0.93


In [11]:
# Load per-problem CSV
per_problem_path = local_paths.get(per_problem_name)
if per_problem_path and per_problem_path.exists():
    per_problem_df = pd.read_csv(per_problem_path)
    display(per_problem_df)
else:
    print(f"Per-problem file not found: {per_problem_name}")

,problem_idx,ground_truth,n_samples,n_correct,accuracy
0,0,18,4,4,1.0
1,1,3,4,4,1.0
2,2,70000,4,2,0.5
3,3,540,4,4,1.0
4,4,20,4,4,1.0
...,...,...,...,...,...
95,95,40,4,4,1.0
96,96,3,4,4,1.0
97,97,12,4,4,1.0
98,98,5,4,4,1.0


## 4. Load Output Parquet & Inspect Individual Problems

In [12]:
# Load the full outputs
parquet_path = local_paths.get(parquet_name)
if parquet_path and parquet_path.exists():
    outputs_df = pd.read_parquet(parquet_path)
    print(f"Loaded {len(outputs_df)} problems from {parquet_name}")
    print(f"Columns: {outputs_df.columns.tolist()}")
else:
    raise FileNotFoundError(f"Parquet not found: {parquet_name}")

Loaded 100 problems from gsm8k_qwen_n4_l32768_outputs.parquet
Columns: ['data_source', 'prompt', 'ability', 'reward_model', 'extra_info', 'responses']


### Pick a problem to inspect

In [ ]:
PROBLEM_IDX = 0  # <-- change this to inspect a different problem

row = outputs_df.iloc[PROBLEM_IDX]

# Extract question text
prompt = row["prompt"]
if isinstance(prompt, list) and len(prompt) > 0 and isinstance(prompt[0], dict):
    question_text = prompt[0].get("content", str(prompt))
else:
    question_text = str(prompt)

# Extract ground truth
reward_model = row.get("reward_model", {})
if isinstance(reward_model, dict):
    ground_truth = reward_model.get("ground_truth", "N/A")
else:
    ground_truth = "N/A"

# Extract responses (may be ndarray)
responses = row["responses"]
if hasattr(responses, "tolist"):
    responses = responses.tolist()
if not isinstance(responses, list):
    responses = [responses]

print(f"=== PROBLEM {PROBLEM_IDX} ===")
print("\n--- QUESTION ---")
print(question_text)
print("\n--- GROUND TRUTH ---")
print(ground_truth)
print(f"\n--- NUMBER OF SAMPLES: {len(responses)} ---")
for i, resp in enumerate(responses):
    print(f"\n--- RESPONSE {i + 1} (last 300 chars) ---")
    text = str(resp)
    print(text[-3000:] if len(text) > 300 else text)
    print("-" * 40)

=== PROBLEM 0 ===

--- QUESTION ---
[{'role': 'user', 'content': 'Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers\' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers\' market? Let\'s think step by step and output the final answer after "####".'}]

--- GROUND TRUTH ---
18

--- NUMBER OF SAMPLES: 4 ---

--- RESPONSE 1 (last 300 chars) ---
es four eggs for the muffins, but the muffins are made with four eggs, so that's 4 eggs. Then, she eats three eggs for breakfast. So, total eggs used are 3 + 4 = 7. Therefore, 16 - 7 = 9. So, 9*2=18. So, the answer is $18.

But let me check again. Maybe the problem is that she eats three eggs for breakfast every morning, but does that mean she eats them once a day? So, if she eats three eggs in the morning, then the rest of the day, she has 16 - 3 = 13 eggs. Then she bakes muffins fo

In [25]:
from IPython.display import display, Markdown
import random

# ---------------------------------------------------------
# CONFIGURE YOUR INSPECTION MODE HERE
# ---------------------------------------------------------
# Option 1: View a specific problem by index
PROBLEM_IDX = 0

# Option 2: Pick a random problem that is fully correct (all samples right)
# Set PICK_MODE = "random_correct" to use this
# Set PICK_MODE = "random_wrong" to pick a problem where at least one sample was wrong
# Set PICK_MODE = "specific" to use PROBLEM_IDX above
PICK_MODE = "random_wrong"  # "specific" | "random_correct" | "random_wrong"

# How many trailing characters to show per response
RESPONSE_TRUNCATE = 3000

# ---------------------------------------------------------
# Resolve which problem to show
# ---------------------------------------------------------
if PICK_MODE == "random_correct":
    # accuracy == 1.0 means all samples for this problem were correct
    correct_mask = per_problem_df["accuracy"] == 1.0
    if not correct_mask.any():
        raise ValueError("No fully-correct problems found in this run.")
    candidates = per_problem_df.loc[correct_mask, "problem_idx"].tolist()
    chosen_idx = random.choice(candidates)
    print(f"[random_correct] Picked problem {chosen_idx} (from {correct_mask.sum()} fully-correct problems)")

elif PICK_MODE == "random_wrong":
    # accuracy < 1.0 means at least one sample failed
    wrong_mask = per_problem_df["accuracy"] < 1.0
    if not wrong_mask.any():
        raise ValueError("No incorrect problems found in this run (perfect score!).")
    candidates = per_problem_df.loc[wrong_mask, "problem_idx"].tolist()
    chosen_idx = random.choice(candidates)
    print(f"[random_wrong] Picked problem {chosen_idx} (from {wrong_mask.sum()} problems with errors)")

else:
    chosen_idx = PROBLEM_IDX

# ---------------------------------------------------------
# Extract data for the chosen problem
# ---------------------------------------------------------
row = outputs_df.iloc[chosen_idx]

# Question text
prompt = row["prompt"]
if isinstance(prompt, list) and len(prompt) > 0 and isinstance(prompt[0], dict):
    question_text = prompt[0].get("content", str(prompt))
else:
    question_text = str(prompt)

# Ground truth
reward_model = row.get("reward_model", {})
if isinstance(reward_model, dict):
    ground_truth = reward_model.get("ground_truth", "N/A")
else:
    ground_truth = "N/A"

# Responses (handle ndarray)
responses = row["responses"]
if hasattr(responses, "tolist"):
    responses = responses.tolist()
if not isinstance(responses, list):
    responses = [responses]

# Accuracy for this problem (from the CSV)
accuracy = per_problem_df.loc[per_problem_df["problem_idx"] == chosen_idx, "accuracy"].values[0]
n_correct = per_problem_df.loc[per_problem_df["problem_idx"] == chosen_idx, "n_correct"].values[0]

# ---------------------------------------------------------
# Display using Markdown (wraps lines + renders LaTeX)
# ---------------------------------------------------------
display(Markdown(f"## Problem {chosen_idx}"))
display(Markdown(f"**Accuracy:** {accuracy:.2f} ({n_correct}/{len(responses)} samples correct)"))
display(Markdown(f"**Ground truth:** {ground_truth}"))
display(Markdown("---"))
display(Markdown("### Question"))
display(Markdown(question_text))

# Show each response in a wrapped, pre-formatted block
for i, resp in enumerate(responses):
    text = str(resp)
    snippet = text[-RESPONSE_TRUNCATE:] if len(text) > RESPONSE_TRUNCATE else text

    display(Markdown(f"### Response {i + 1} ({len(text)} chars, last {len(snippet)} shown)"))

    # Use HTML pre with pre-wrap so long lines wrap but newlines are preserved
    from html import escape
    html = (
        f'<div style="background:#f8f8f8; padding:10px; border-radius:4px; '
        f'border:1px solid #ddd; font-family:monospace; font-size:13px;">'
        f'<pre style="white-space:pre-wrap; word-wrap:break-word; margin:0;">'
        f'{escape(snippet)}</pre></div>'
    )
    display(Markdown(html))

[random_wrong] Picked problem 29 (from 15 problems with errors)


## Problem 29

**Accuracy:** 0.25 (1/4 samples correct)

**Ground truth:** 104

---

...

<div style="background:#f8f8f8; padding:10px; border-radius:4px; border:1px solid #ddd; font-family:monospace; font-size:13px;"><pre style="white-space:pre-wrap; word-wrap:break-word; margin:0;"> and the two pairs of heels is five dollars less than the two pairs of heels. Which would mean that the boots are five dollars less than the two pairs of heels. So, B = 99 - 5 = 94. Even though the problem statement is contradictory, this might be the intended answer.

So, if we assume that the problem meant that the total cost of the boots and the two pairs of high heels is five dollars less than the two pairs of high heels, then B = 94. So, the boots cost $94.

But the problem says that the boots and the two pairs of high heels together cost five dollars less than the boots. Which would be B + 99 = B - 5, which is impossible. But if we ignore the contradiction and proceed with the other interpretation, then the answer is $94.

Alternatively, maybe the problem meant that the boots are five dollars less than the two pairs of high heels, and the total of the boots and the two pairs of high heels is five dollars less than the two pairs of high heels. Which would mean that B = 99 - 5 = 94, and B + 99 = 99 - 5 → 193 = 94. Which is not true. So, no.

But since the problem provides the prices of the two pairs of heels, and asks for the boots, and the only way to make sense of it is if the boots are $94, I think that might be the intended answer, despite the contradiction in the problem statement. So, I&#x27;ll go with that.

**Final Answer**
The boots cost \boxed{94} dollars.
&lt;/think&gt;

To solve the problem, we need to determine the cost of the boots based on the given information.

---

### Step 1: Understand the given data

We are told:

- One pair of heels costs **$33**.
- The other pair of heels costs **twice as much**, so it&#x27;s **$66**.
- The total cost of the **two pairs of high heels** is:
  $$
  33 + 66 = 99
  $$

The problem also states:

&gt; &quot;the boots and two pairs of high heels that together cost five dollars less than the boots.&quot;

This is a key point. Let’s denote the cost of the boots as **$B**.

So, the total cost of the boots and the two pairs of high heels is:
$$
B + 99
$$

The problem says this total is **five dollars less than the boots**, meaning:
$$
B + 99 = B - 5
$$

But this equation simplifies to:
$$
99 = -5
$$
Which is **not possible**. This contradiction suggests that the original phrasing may have an error or misinterpretation.

---

### Step 2: Reinterpret the problem

Given that the problem is likely intended to be solvable, we consider a more reasonable interpretation.

Assume the correct phrasing is:
&gt; &quot;the boots cost five dollars less than the two pairs of high heels.&quot;

So:
$$
B = 99 - 5 = 94
$$

This would make the total cost of the boots and the two pairs of heels:
$$
94 + 99 = 193
$$

This is **not** five dollars less than the boots, but this is the only way the problem makes sense in terms of the given values.

---

### Step 3: Final Answer

Given the contradiction in the original phrasing, but the only logical conclusion that fits the values provided, we conclude that the **cost of the boots is**:

$$
\boxed{94}
$$&lt;|im_end|&gt;</pre></div>

### Response 3 (14335 chars, last 3000 shown)

<div style="background:#f8f8f8; padding:10px; border-radius:4px; border:1px solid #ddd; font-family:monospace; font-size:13px;"><pre style="white-space:pre-wrap; word-wrap:break-word; margin:0;">g that the boots and the two pairs of high heels together cost five dollars less than the boots. Which would mean that the total is five dollars less than the boots. So, if the total is (boots) - 5, then:

Boots + (heels) = Boots - 5

Which implies that (heels) = -5, which is impossible. Therefore, there must be a different interpretation.

Wait, perhaps the problem is saying that the boots and the two pairs of high heels together cost five dollars less than the boots. So, the total is five dollars less than the boots. So, if the boots cost X, then X + (heels) = X - 5. Which is impossible. Therefore, there&#x27;s a contradiction here, which suggests that maybe the problem is misworded.

Alternatively, maybe the problem is saying that the boots and the two pairs of high heels together cost five dollars less than the boots. Which would mean that the total is five dollars less than the boots. Which would mean that the total is X - 5, and the total is also X + (heels). So:

X + (heels) = X - 5

Which implies that (heels) = -5, which is impossible.

Alternatively, maybe the problem is saying that the two pairs of high heels cost five dollars less than the boots. So, the two pairs of high heels cost (X - 5). But the two pairs of high heels are $33 and $66, which sum to $99. Therefore, X - 5 = 99, so X = 104. Therefore, the boots cost $104. This seems like a possible answer, even though the problem&#x27;s wording is confusing. Maybe the original problem had a different phrasing.

Given that, and since the problem mentions that one pair of heels costs $33 and the other costs twice as much, which is $66, totaling $99, and if the two pairs of high heels cost five dollars less than the boots, then the boots are $104. Therefore, the answer is $104.

So, even though the problem&#x27;s wording is confusing, this seems to be the only way to make sense of it. Therefore, the boots cost $104.

**Final Answer**
The boots cost \boxed{104} dollars.
&lt;/think&gt;

To determine the cost of the boots, we are given the following key information:

- One pair of heels costs **$33**.
- The other pair of heels costs **twice as much**, so it costs **$66**.
- Together, these two pairs of heels cost **$33 + $66 = $99**.
- The boots and the two pairs of heels **together cost five dollars less than the boots**.

Let’s denote the cost of the boots as **$ x $**.

According to the problem, the total cost of the boots and the two pairs of heels is **$ x + 99 $**.

The problem states that this total is **five dollars less than the boots**, i.e.:

$$
x + 99 = x - 5
$$

This equation is problematic because it implies $ 99 = -5 $, which is not possible. Therefore, there must be a misinterpretation or misstatement in the original problem.

However, a more reasonable interpretation is that the **two pairs of heels together cost five dollars less than the boots**. That is:

$$
x - 99 = 5
$$

Solving this:

$$
x = 99 + 5 = 104
$$

So, the boots cost **$104**.

---

### Final Answer:

$$
\boxed{104}
$$&lt;|im_end|&gt;</pre></div>

### Response 4 (16789 chars, last 3000 shown)

<div style="background:#f8f8f8; padding:10px; border-radius:4px; border:1px solid #ddd; font-family:monospace; font-size:13px;"><pre style="white-space:pre-wrap; word-wrap:break-word; margin:0;">gether cost five dollars less than the two pairs of high heels. Which would mean:

B + 99 = 99 - 5 =&gt; B = -5. Also impossible.

Alternatively, maybe the problem is that the boots cost five dollars less than the two pairs of high heels. So, B = 99 - 5 = 94. Then, the answer is 94. So, even though the problem says &quot;the boots and two pairs of high heels that together cost five dollars less than the boots,&quot; which is impossible, but maybe it&#x27;s a misstatement, and the actual intended meaning is that the boots cost five dollars less than the two pairs of high heels. In that case, the answer is 94.

Alternatively, maybe the problem is that the total cost of the boots and the two pairs of high heels is five dollars less than the total cost of the two pairs of high heels. Which would mean that B + 99 = 99 - 5 =&gt; B = -5. Which is impossible.

Alternatively, maybe the problem is that the total cost of the boots and the two pairs of high heels is five dollars less than the total cost of the boots. Which is the same as before.

Alternatively, maybe the problem is that the boots cost five dollars less than the two pairs of high heels. So, B = 99 - 5 = 94. So, the answer is 94.

Given that, even though the wording is confusing, I think that&#x27;s the intended meaning. So, the answer is 94.

**Final Answer**
The boots cost \boxed{94} dollars.
&lt;/think&gt;

To solve the problem, let&#x27;s break it down step by step.

---

### Step 1: Understand the Given Information

- One pair of heels costs **$33**.
- The other pair of heels costs **twice as much**, so it is $33 × 2 = $66.
- The **total cost of the two pairs of heels** is therefore:
  $$
  33 + 66 = 99 \text{ dollars}
  $$

- The problem states that **the boots and the two pairs of high heels together cost five dollars less than the boots**.

This is a key point. Let’s denote the cost of the boots as $ B $.

---

### Step 2: Set Up the Equation

The problem says:

&gt; &quot;The boots and the two pairs of high heels together cost five dollars less than the boots.&quot;

So, the total cost of the boots and the two pairs of heels is five dollars less than the cost of the boots:

$$
B + 99 = B - 5
$$

This equation simplifies to:

$$
99 = -5
$$

This is a **contradiction**, which suggests that the wording is **not consistent**. However, this is a common type of wordplay in problems like this, where the phrasing might be misleading.

---

### Step 3: Interpret the Problem Differently

If we interpret the statement **&quot;the boots and the two pairs of high heels together cost five dollars less than the boots&quot;** as:

&gt; The **total cost of the boots and the two pairs of high heels is five dollars less than the cost of the two pairs of high heels**, then:

$$
B + 99 = 99 - 5
$$

$$
B = -5
$$

Again, this is impossible. So, the only logical interpretation that makes sense is:

&gt; The **boots cost five dollars less than the total cost of the two pairs of high heels**.

That is:

$$
B = 99 - 5 = 94
$$

---

### Final Answer

$$
\boxed{94}
$$&lt;|im_end|&gt;</pre></div>

### Side-by-side comparison of all samples for one problem

In [16]:
# Set this to the problem you want to compare across samples
COMPARE_PROBLEM_IDX = 0
SHOW_FULL_RESPONSE = False  # Set True to show full text (can be very long)
TRUNCATE_LEN = 500

row = outputs_df.iloc[COMPARE_PROBLEM_IDX]
responses = row["responses"]
if hasattr(responses, "tolist"):
    responses = responses.tolist()
if not isinstance(responses, list):
    responses = [responses]

comparison_rows = []
for i, resp in enumerate(responses):
    text = str(resp)
    display_text = text if SHOW_FULL_RESPONSE else (text[-TRUNCATE_LEN:] if len(text) > TRUNCATE_LEN else text)
    comparison_rows.append({
        "Sample": i + 1,
        "Length": len(text),
        "Last chars": display_text,
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

# Also show the prompt for reference
prompt = row["prompt"]
if isinstance(prompt, list) and len(prompt) > 0:
    print("\n=== PROMPT ===")
    print(prompt[0].get("content", str(prompt)))

,Sample,Length,Last chars
0,1,6863,ep 4: Eggs Remaining for Sale\nTo find out how...
1,2,4996,ber of eggs used per day for breakfast and muf...
2,3,5348,for Baking Muffins\nJanet bakes muffins for he...
3,4,4429,"fast every morning, which is once a day.\n3. *..."


## 5. Optional: Re-score a single response locally (for debugging)

If you want to test how the scorer behaves on a specific response string, run the cell below.

In [ ]:
# Uncomment and edit the response you want to test:

# test_response = responses[0]  # use the first response from the problem above
# test_ground_truth = ground_truth

# from verl.utils.reward_score.curriculum_math.compute_score import compute_score
# score = compute_score(test_response, test_ground_truth)
# print(f"Score: {score}")